In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, classification_report


In [ ]:
# Import the dataset from csv file into data frames 
cust_satisfaction_df = pd.read_csv('/Users/ravvarma/Documents/Personal/AI & ML IITM /Week-21-Graded_Project/Customer Satisfaction.csv')
cust_satisfaction_df.head(10)
# Summary of Data set before cleaning 
print(cust_satisfaction_df.describe())
print(cust_satisfaction_df.shape)

# Check for missing values, data types and duplicates in the dataset
print(f"Missing values in Customer Satisfaction data frame is :\n{cust_satisfaction_df.isna()}")
print(f"The data type of data in Customer Satisfaction df is :\n{cust_satisfaction_df.dtypes}")
print(f"Duplicates in Customer Satisfaction df is :\n{cust_satisfaction_df[cust_satisfaction_df.duplicated]}")


In [ ]:
# 1. Gender Distribution
# 1.a What is the proportion of male to female travelers in the dataset?

# Calculate the ratio of value in the Gender column 
# gender_ratio= round(cust_satisfaction_df['Gender'].value_counts(normalize=True),2)*100
gender_ratio= cust_satisfaction_df['Gender'].value_counts(normalize=True)
print(f"The proportion of Female to Male travellers in the dataset is  :\n"
      + gender_ratio.to_string(header=False))

# Chart depicting Gender Ratio
plt.figure(figsize=(8, 6))
plt.pie(gender_ratio, 
        labels=gender_ratio.index, 
        autopct='%1.1f%%',  # Show percentages on the chart with 1 decimal place
        startangle=90, 
       )
plt.title('Gender Distribution')
plt.axis('equal')  # Equal aspect ratio ensures the pie chart is circular
plt.tight_layout()
plt.show()


In [ ]:
# 1.b How does gender relate to satisfaction levels 
satisfaction_df = round(cust_satisfaction_df.groupby('Gender')['Satisfaction'].
                        value_counts(normalize=True),2)*100
print(f"Gender wise satisfaction % \n"+ satisfaction_df.to_string(header=False))

# Chart depicting satisfaction level by gender
# Unstack the DataFrame to create separate Series for each 'Gender'
satisfaction_df_unstacked = satisfaction_df.unstack('Gender')
# Create a figure with two subplots (one for each 'Gender')
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
# Plot a pie chart for each 'Gender'
for i, gender in enumerate(satisfaction_df_unstacked.columns):
    axs[i].pie(satisfaction_df_unstacked[gender].dropna(), 
               labels=satisfaction_df_unstacked.index, autopct='%1.1f%%', startangle=90)
    axs[i].set_title(gender)

# Layout so plots do not overlap
fig.tight_layout()
plt.show()


In [ ]:
# 2. Age Analysis 
# 2.1 Average age of travellers
AverageAge = cust_satisfaction_df['Age'].mean(skipna=True)
print(f"Average age of the travellers is {AverageAge:.2f}")

In [ ]:
# 2.2 What is the age range of the travelers, and how many travelers fall within specific age brackets (e.g., 18-30, 31-45, 46-60, 61+)?
# To find the age range of the travelers and the number of travelers in specific age brackets, we will bin the 'Age' column as follows : 

bins = [18, 30, 45, 60, float('inf')]
labels = ['18-30', '31-45', '46-60', '61+']
age_brackets = pd.cut(cust_satisfaction_df['Age'], bins=bins, labels=labels)
print(age_brackets.value_counts())

# plot the age bracket using bar chart 
age_brackets.value_counts().plot(kind='bar')
plt.title('Age Distribution')
plt.xlabel('Age Bracket')
plt.ylabel('Count')
plt.show()


In [ ]:
# 3. Travel Category
# 3.1 How many travelers are classified under "Official" versus "Personal Travel"?

travel_category_df = cust_satisfaction_df[cust_satisfaction_df['Travel Category'].isin(['Official', 'Personal Travel'])].groupby('Travel Category').size()
print(f"Traveler Classification under Official vs Personal travel \n"+ travel_category_df.to_string())

#plot the share of "Official" versus "Personal Travel" category travellers 
travel_category_df.plot(kind='bar')
plt.title('Official vs Personal travel Distribution')
plt.xlabel('Travel Category')
plt.ylabel('Number of passengers')
plt.show()

In [ ]:
# 3.2 What is the average distance traveled by each travel category?
average_distance = cust_satisfaction_df.groupby('Travel Category')['Distance Travelled'].mean()
print(f"Average distance traveled by each travel category:\n{average_distance.round(2)}")

# Plot average distance traveled by each travel category
average_distance.plot(kind='bar')
plt.title('Average Distance travelled by each Travel Category')
plt.xlabel('Travel Category')
plt.ylabel('Average Distance')
plt.show()

In [ ]:
# 4. Travel Class ratings 
# 4.1 What is the average rating for seat comfort across different travel classes?

average_rating = cust_satisfaction_df.groupby('Travel Class')['Seat Comfort'].mean()
print(f"Average rating for seat comfort across different travel classes:\n{average_rating.round(2)}")

# Plot average rating of seat comfort by each travel class
average_rating.plot(kind='bar')
plt.title('Average Seat comfort rating by Travel Class')
plt.xlabel('Travel Class')
plt.ylabel('Average Rating by Seat Comfort')
plt.show()

In [ ]:
# 4.2 How does the rating for food differ between "Business" and "Eco" classes?

Food_Rating_df = cust_satisfaction_df[cust_satisfaction_df['Travel Class'].isin(['Business', 'Economy'])].groupby(['Travel Class','Food']).size()
print(f"Rating for food between 'Business' and 'Economy' classes \n"+ Food_Rating_df.to_string())

#plot the Rating for food between 'Business' and 'Economy' classes
Food_Rating_df.unstack().plot(kind='bar')
plt.title('Food Rating Distribution')
plt.xlabel('Travel Class')
plt.ylabel('Count')
plt.show()

In [ ]:
# 5. Delay Analysis 
# 5.1 What is the average departure delay and arrival delay for the dataset?
average_dep_delay = cust_satisfaction_df['Departure Delay (min)'].mean()
print(f"Average Departure Delay:\n{average_dep_delay.round(2)}")

average_arrival_delay = cust_satisfaction_df['Arrival Delay (min)'].mean()
print(f"Average Arrival Delay:\n{average_arrival_delay.round(2)}")

average_dep_delay = cust_satisfaction_df['Departure Delay (min)'].mean()
average_arrival_delay = cust_satisfaction_df['Arrival Delay (min)'].mean()

# Plot the average Arrival vs Departure delays in a 
delays = pd.Series([average_dep_delay, average_arrival_delay], index=['Departure Delay', 'Arrival Delay'])
delays.plot(kind='pie', autopct='%1.1f%%')
plt.title('Average Delay')
plt.show()

In [ ]:
# 5.2 How do delays (both departure and arrival) impact customer satisfaction?
print(f"Impact on Customer Satisfaction due to Departure and Arrival delays \n")

plt.scatter(cust_satisfaction_df['Departure Delay (min)'], cust_satisfaction_df['Satisfaction'].map({'satisfied': 1, 'dissatisfied': 0}))
plt.title('Departure Delay vs Satisfaction')
plt.xlabel('Departure Delay (min)')
plt.ylabel('Satisfaction')
plt.show()

plt.scatter(cust_satisfaction_df['Arrival Delay (min)'], cust_satisfaction_df['Satisfaction'].map({'satisfied': 1, 'dissatisfied': 0}))
plt.title('Arrival Delay vs Satisfaction')
plt.xlabel('Arrival Delay (min)')
plt.ylabel('Satisfaction')
plt.show()

In [ ]:
print("""Model Building - 

As part of Model Building, we are performing the following activities in the following order -
1. Data Cleaning
    a. Check for duplicate rows
    b. Handle missing values 
    c. Ensure right data type for analysis
    d. Handle outliers
2. Feature Engineering          
    a. Create new features based on existing ones
    b. Scale numerical features if necessary
3. Exploratory Data Analysis (EDA)
    a. Visualise the distribution of the target variable
    b. Investigate correlations between features and satisfaction
    c. Identify trends or outliers in the data
4. Data Preparation              
    1. Split the dataset into features (X) and target variable (y)
    2. Encode categorical variables into numerical format.
5. Model Training
    a. Split the data into training and testing sets.
    b. Train multiple models using different algorithms (e.g., Decision Tree, KNN etc.)
    c. Evaluate models using metrics like accuracy, precision, recall, and F1-score.""")

print("""
1. Data Cleaning                
As part of data cleaning we will do the following : 
   1. Check for duplicate rows 
   2. Handle missing values 
   3. Ensure right data type for analysis 
   4. Handle outliers 
I am implementing the strategy of neither removing the missing values, nor ignoring the missing values as part of model build. 
The reason for such a strategy is - 
Removing the missing values could cause biase in the model. Moreover, the number of rows having missing values is very large. For example - 
    1. Travel Class is a key column and has close to 6283 rows with null values. 
    2. Food Rating - Close to 13958 rows have null values. 
Removing all these rows could result in biased or incorrect prediction 
A hard coded value of 'Unknown' ia assigned to missing to categorical values and mean is assigned to missing numerical values. 
""")

In [ ]:
# Identify column categories
cat_col = [col for col in cust_satisfaction_df.columns if cust_satisfaction_df[col].dtype == 'object']
num_col = [col for col in cust_satisfaction_df.columns if cust_satisfaction_df[col].dtype != 'object']

print('Categorical columns:', cat_col)
print('Numerical columns:', num_col)

# Summarize the number of rows with missing values for each column

for i in range(cust_satisfaction_df.shape[1]):
  missing_rows = cust_satisfaction_df.iloc[:, i].isnull().sum()
  perc_of_missing_rows = missing_rows / cust_satisfaction_df.shape[0] * 100
  print('%d, Number of missing values in the row: %d (%.1f%%)' %(i, missing_rows, perc_of_missing_rows))

print("""
Decide to whether drop the columns that have very high number of missing values or not 
As the maximum % of rows having missing values is max 14%, deciding not to drop such columns. Sample output for reference - 

        0, Number of missing values in the row: 0 (0.0%)
        1, Number of missing values in the row: 0 (0.0%)
        2, Number of missing values in the row: 0 (0.0%)
        3, Number of missing values in the row: 0 (0.0%)
        4, Number of missing values in the row: 6283 (6.0%)
        5, Number of missing values in the row: 0 (0.0%)
        6, Number of missing values in the row: 12170 (11.7%)
        7, Number of missing values in the row: 12149 (11.7%)
        8, Number of missing values in the row: 9209 (8.9%)
        9, Number of missing values in the row: 13958 (13.4%)
        10, Number of missing values in the row: 10136 (9.8%)
        11, Number of missing values in the row: 8231 (7.9%)
        12, Number of missing values in the row: 15272 (14.7%)
        13, Number of missing values in the row: 8253 (7.9%)
        14, Number of missing values in the row: 0 (0.0%)
        15, Number of missing values in the row: 0 (0.0%)
        16, Number of missing values in the row: 310 (0.3%)
        17, Number of missing values in the row: 0 (0.0%) 
 
The data set has both numeric(departure, boarding, food) and categorical features (travel class). 
Both numeric and categorical features have null values. As part of the cleaning process - 
    a. Assign "Unknown" value for Null rows in categorical column - "Travel Class"
    b. Take the median of individual columns and assign the mean to Null rows in Numerical columns like - "Departure", "Boarding", "Food" etc 
""")
#a. Assign "Unknown" value for Null rows in categorical column - "Travel Class"
if 'Travel Class' in cust_satisfaction_df.columns:
    cust_satisfaction_df['Travel Class'] = cust_satisfaction_df['Travel Class'].fillna('Unknown')
    print("The 'Travel Class' column Exists in the DataFrame.")
else:
    print("The 'Travel Class' column does not exist in the DataFrame.")

# --> b. Take the median of individual columns and assign the mean to Null rows in Numerical columns like - "Departure", "Boarding", "Food" etc 
cust_satisfaction_df['Departure/Arrival Rating'] = cust_satisfaction_df['Departure/Arrival Rating'].fillna(cust_satisfaction_df['Departure/Arrival Rating'].mean())
cust_satisfaction_df['Booking Ease'] = cust_satisfaction_df['Booking Ease'].fillna(cust_satisfaction_df['Booking Ease'].mean())
cust_satisfaction_df['Boarding Point'] = cust_satisfaction_df['Boarding Point'].fillna(cust_satisfaction_df['Boarding Point'].mean())
cust_satisfaction_df['Food'] = cust_satisfaction_df['Food'].fillna(cust_satisfaction_df['Food'].mean())
cust_satisfaction_df['Seat Comfort'] = cust_satisfaction_df['Seat Comfort'].fillna(cust_satisfaction_df['Seat Comfort'].mean())
cust_satisfaction_df['Entertainment'] = cust_satisfaction_df['Entertainment'].fillna(cust_satisfaction_df['Entertainment'].mean())
cust_satisfaction_df['Leg Room'] = cust_satisfaction_df['Leg Room'].fillna(cust_satisfaction_df['Leg Room'].mean())
cust_satisfaction_df['Luggage Handling'] = cust_satisfaction_df['Luggage Handling'].fillna(cust_satisfaction_df['Luggage Handling'].mean())
cust_satisfaction_df['Arrival Delay (min)'] = cust_satisfaction_df['Arrival Delay (min)'].fillna(cust_satisfaction_df['Arrival Delay (min)'].mean())

In [ ]:
## 3. Ensure right data type for analysis 
# -----------------------------------------
print(f"The data types of various columns are -")
print(cust_satisfaction_df.dtypes)
## Almost all of the ratings are having float data type. Since we dont have any ratings which have a decimal in them, 
# converting all float from Float to Int. 

cust_satisfaction_df['Departure/Arrival Rating'] = cust_satisfaction_df['Departure/Arrival Rating'].astype(int)
cust_satisfaction_df['Booking Ease'] = cust_satisfaction_df['Booking Ease'].astype(int)
cust_satisfaction_df['Boarding Point'] = cust_satisfaction_df['Boarding Point'].astype(int)
cust_satisfaction_df['Food'] = cust_satisfaction_df['Food'].astype(int)
cust_satisfaction_df['Seat Comfort'] = cust_satisfaction_df['Seat Comfort'].astype(int)
cust_satisfaction_df['Entertainment'] = cust_satisfaction_df['Entertainment'].astype(int)
cust_satisfaction_df['Leg Room'] = cust_satisfaction_df['Leg Room'].astype(int)
cust_satisfaction_df['Luggage Handling'] = cust_satisfaction_df['Luggage Handling'].astype(int)
cust_satisfaction_df['Arrival Delay (min)'] = cust_satisfaction_df['Arrival Delay (min)'].astype(int)

In [ ]:
## 4. Handle outliers 
# ---------------------- 
## In order to identify and remove the outliers we will use the IQR method. We will define a function which will remove the ouliers 
## from a column if they are less than 5%. Else they wont be removed. 
def fn_remove_ouliers(df, column_name, threshold_percent=5):

# 1. Calculate Q1, Q3 and IQR
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

# 2. Identify outliers
    outliers = df[(df[column_name] < lower_bound) | (df[column_name] > upper_bound)]
    num_outliers = len(outliers)
    total_rows = len(df)

# 3. Calculate percentage of outliers
    percent_outliers = (num_outliers / total_rows) * 100

# 4. Apply conditional logic
    if percent_outliers ==0:
        print(f"Column-'{column_name}': Outlier percentage ({percent_outliers:.2f}%) is 0. No action performed")
        return df
    elif percent_outliers < threshold_percent:
        print(f"Column-'{column_name}': Outlier percentage ({percent_outliers:.2f}%) is below {threshold_percent}%. Removing outliers.")
        
        df_filtered = df[(df[column_name] >= lower_bound) & (df[column_name] <= upper_bound)]
        return df_filtered
    else:
        print(f"Column-'{column_name}': Outlier percentage ({percent_outliers:.2f}%) is {threshold_percent}% or more. No action performed.")
        return df

## Iteratively call this function for all columns of the Customer Satisfaction dataframe 
column_list = ['Distance Travelled','Departure/Arrival Rating','Booking Ease','Boarding Point','Food','Seat Comfort','Entertainment',
               'Leg Room','Luggage Handling','Cleanliness','Departure Delay (min)','Arrival Delay (min)'] 

cleaned_cust_satisfaction_df= cust_satisfaction_df.copy()

for x in column_list:
  cleaned_cust_satisfaction_df = fn_remove_ouliers(cleaned_cust_satisfaction_df, x, threshold_percent=5)


In [ ]:
print("""
2. Data Preparation              
    1. Split the dataset into features (X) and target variable (y)
    2. Encode categorical variables into numerical format.

Response ---> 
The splitting of dataset into X and y variables is covered in Section - "5. Model Training"
#   1. Split the data into training and testing sets. The steps involved in splitting the data are in the following sequence - 
#       a. Split the data 
#       b. Encode the training data 
#       c. Transform both training and testing data sets   

#Please refer to "5. Model Training" for more details. 
""")

In [ ]:
print("""3. Feature Engineering          
As part of feature engineering we will do the following : 
    1. Create new features based on existing ones 
    2. Scale numerical features if necessary. 
""")
# 1. Create new features based on existing ones 
#    I am creating 3 types of new Features - Pre_Flight_Experience_rating, In_Flight_Experience_Rating, Total_Delay 

Pre_Flight_Experience_columns = ['Departure/Arrival Rating','Booking Ease','Boarding Point']
cleaned_cust_satisfaction_df['Pre_Flight_Experience_rating']= cleaned_cust_satisfaction_df[Pre_Flight_Experience_columns].sum(axis=1)

In_Flight_Experience_columns = ['Food','Seat Comfort','Entertainment','Leg Room','Luggage Handling', 'Cleanliness']
cleaned_cust_satisfaction_df['In_Flight_Experience_Rating']= cleaned_cust_satisfaction_df[In_Flight_Experience_columns].sum(axis=1)
cleaned_cust_satisfaction_df['In_Flight_Experience_Rating'].head()

Total_Delay_columns = ['Departure Delay (min)', 'Arrival Delay (min)']
cleaned_cust_satisfaction_df['Total_Delay']= cleaned_cust_satisfaction_df[Total_Delay_columns].sum(axis=1)
    
#   2. Scale numerical features if necessary. 
#       Numerical features are being sclaed as part of encoding. I am using standard scaler from scikit libraries to scale the numerical data. 
#       Please refer to the code section (Model Training) for details.  


In [ ]:
print("""4. Exploratory Data Analysis (EDA)  
    1. Visualise the distribution of the target variable.
    2. Investigate correlations between features and satisfaction.
    3. Identify trends or outliers in the data.
""")

#   1. Visualise the distribution of the target variable
#       a. The target variable 'Satisfaction' is a categorical variable. We will use both Matplotlib and Seaborn plots to check its ditribution across the target variables 
#       b. We will check distribution against original features like Total Passengers (both % and count), and Travel Class
#       c. We will also check satisfaction level across 3 phases of travel - Pre flight, In Flight and Overall delay. These are new features created.

#   Satisfaction % across total passengers
cleaned_cust_satisfaction_df['Satisfaction'].value_counts().plot.pie(
    figsize=(5, 5),
    autopct='%1.1f%%',
    startangle=90
) 
plt.title("Satisfaction % across total passengers")
plt.tight_layout()
plt.show()

#   Following barplot also shows the count of passengers that are Satisfied vs who are Unsatisfied
cleaned_cust_satisfaction_df['Satisfaction'].value_counts().plot(
    kind='bar',
    figsize=(5, 5)
)
plt.title('Customer Satisfaction by passenger count ')
plt.xlabel('Satisfaction')
plt.ylabel('Number of Customers')
plt.tight_layout()
plt.show()

#   Satisfaction % by Travel Class of passengers 
plt.figure(figsize=(5,5))
sns.countplot(
    x='Travel Class',
    hue='Satisfaction',
    data=cleaned_cust_satisfaction_df
)
plt.title("Satisfaction by Travel Class of Passengers")
plt.tight_layout()
plt.show()

#   Satisfaction level across 3 phases of travel - Pre flight, In Flight and Overall delay. These are the new features created.
fig, axs = plt.subplots(1, 2, figsize=(15, 5))
sns.boxenplot(x="Pre_Flight_Experience_rating", y="Satisfaction", data=cleaned_cust_satisfaction_df, ax=axs[0])
sns.boxenplot(x="In_Flight_Experience_Rating", y="Satisfaction", data=cleaned_cust_satisfaction_df, ax=axs[1])
plt.tight_layout()
plt.show()

plt.figure(figsize=(5,5))
sns.boxenplot(x="Total_Delay", y="Satisfaction", data=cleaned_cust_satisfaction_df)
plt.show()

cleaned_cust_satisfaction_df.head(10)

In [ ]:
# 2.Investigate correlations between features and satisfaction.
#    a. Correlations between featuires and satisfaction can be obtained by calculating the Pearson Correlation coefficients 
#    b. Finally, the coefficients will be plotted using a Heat map which will show the relationshiops between all variables in the dataset

correlation_matrix = cleaned_cust_satisfaction_df.corr(numeric_only=True)
print(f"The cooorelation coefficients will be in the range of -1 to 1")
print(correlation_matrix)
plt.figure(figsize=(12, 12))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Features and Satisfaction')
plt.show()

print("""
Observations from the Correlation matrix - 
    1. Food, Seat Comfort and Entertainment and Cleanliness have a strong positive coorelation. These features tend to increase togetgher. 
    2. Booking ease, Boarding point are negatively corelated to Departur/Arrival Rating. When one variable increases, the other decreases.
    3. The engineered feature "Pre Flight Experience Rating" and Departure/Arrival rating, Booking ease, Boarding Point have a strong positive coorelation.
    4. Similarly the engineered feature "Inflight Experience Rating" and the features from which it was built like (Food, Seat Comfort, Entertainment etc) have a strong positive coorelation. 
""")

In [ ]:
# 3. Identify trends or outliers in the data in the order of maximum outliers

outlier_percentages = []
for x in column_list:
    Q1 = cleaned_cust_satisfaction_df[x].quantile(0.25)
    Q3 = cleaned_cust_satisfaction_df[x].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = cleaned_cust_satisfaction_df[(cleaned_cust_satisfaction_df[x] < lower_bound) | (cleaned_cust_satisfaction_df[x] > upper_bound)]
    num_outliers = len(outliers)
    total_rows = len(cleaned_cust_satisfaction_df)

    percent_outliers = (num_outliers / total_rows) * 100

    outlier_percentages.append((x, percent_outliers, lower_bound, upper_bound))

outlier_percentages.sort(key=lambda x: x[1], reverse=True)

for x, percent_outliers, lower_bound, upper_bound in outlier_percentages:
    if percent_outliers == 0:
        print(f"Column-'{x}': Outlier percentage is ({percent_outliers:.2f}%)")
    elif percent_outliers < 5:
        print(f"Column-'{x}': Outlier percentage is ({percent_outliers:.2f}%)")
        df_filtered = cleaned_cust_satisfaction_df[(cleaned_cust_satisfaction_df[x] >= lower_bound) & (cleaned_cust_satisfaction_df[x] <= upper_bound)]
    else:
        print(f"Column-'{x}': Outlier percentage is ({percent_outliers:.2f}%)")

#   Plotting the outliers for all columns

def plot_outliers_subplots(dataframe):
    # Select only numeric columns 
    numeric_cols = dataframe.select_dtypes(include=[np.number]).columns
    num_cols = len(column_list)
    num_rows = (num_cols + 1) // 2  
    num_cols_grid = 4 
    fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols_grid, figsize=(12, 5 * num_rows))
    if num_rows > 1:
        axes = axes.flatten()
    else:
        axes = [axes] if num_cols > 0 else []

    for i, col in enumerate(column_list):
        if i < len(axes): # Ensure we don't go out of bounds of axes list
            ax = axes[i]
            # Use seaborn boxplot for outlier visualization
            sns.boxplot(y=dataframe[col], ax=ax)
            ax.set_title(f'Box Plot of {col}')
            ax.set_ylabel(col)
            ax.grid(True, linestyle='--', alpha=0.6)
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_outliers_subplots(cust_satisfaction_df)

print("""
Observations from the Outlier analysis - 
    1. The box plot corroborates the outlier % calculation as shown above
    2. Almost all of the numerical features fall within the middle 50% of the data (from the 25th percentile, Q1, to the 75th percentile, Q3) 
    3. Majority of the data for these numerical features fall within the whiskers. 
    4. The maximum outliers are found in the Departure Delay and Arrival Delay features. 
""")

In [ ]:
print("""
5. Model Training
    1. Split the data into training and testing sets.
    2. Train multiple models using different algorithms (e.g., Decision Tree, KNN etc.)
    3. Evaluate models using metrics like accuracy, precision, recall, and F1-score.
""")
print("""
1. Split the data into training and testing sets.
   The steps involved in splitting the data are in the following sequence - 
       a. Split the data 
       b. Encode the training data 
       c. Transform both training and testing data sets   
""")

#   a. Split the data 
X = cleaned_cust_satisfaction_df.drop(columns='Satisfaction', axis=1)
y = cleaned_cust_satisfaction_df['Satisfaction']

#   a. Splitting the data into training and test sets. 
#       1. We will keep 80% of data as Training data set and 20% of data as Test data set
#       2. We are using stratify as Y to keep the same class distribution for bothj train and test data sets 
#       3. test_size 0.2 denoted 20% of the data set is test data 
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42 )

#   b. Encode the training data. We will use,
#       1. One hot encoding for categorical features , 
#       2. Standard Scaler the numerical values using Standard Scaler from skikit

#   Identify the categorical columns
categorical_columns = cleaned_cust_satisfaction_df.select_dtypes(include=['object']).columns.tolist()
print(categorical_columns)
for col in categorical_columns:
    if cleaned_cust_satisfaction_df[col].dtype == 'object':  
        continue
    else:
        cleaned_cust_satisfaction_df[col] = cleaned_cust_satisfaction_df[col].astype(str)

#   Initialise encoder
encoder = OneHotEncoder(sparse_output=False)

#   Define preprocessing steps for numerical and categorical columns
numerical_columns = cleaned_cust_satisfaction_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_columns = cleaned_cust_satisfaction_df.drop(columns='Satisfaction', axis=1).select_dtypes(include=['object']).columns.tolist()

#   c. Transform both training and testing data sets 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ]
)

In [ ]:
# Fit and transform the data
# Fit the preprocessor ONLY on the training data, then transform both sets

X_train_encoded = preprocessor.fit_transform(X_train) # Fit and transform on train
X_test_encoded = preprocessor.transform(X_test)       # ONLY transform on test

# Convert the results back to a DataFrame for easier viewing (optional)
# Note: fit_transform returns a sparse matrix by default for OneHotEncoder
feature_names = preprocessor.get_feature_names_out()
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=feature_names)
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=feature_names)

In [ ]:
print("""
2. Train multiple models using different algorithms (e.g., Decision Tree, KNN etc.)
   Models to be trained are : 
       a. Logistic Regression
       b. Decision Tree Classifier
       c. K Nearest Neighbour (KNN) Classifier
       d. Random Forest Classifier 
      
3. Evaluate models using metrics like accuracy, precision, recall, and F1-score.
""")

In [ ]:
#  a. Logistic Regression Model  
print("Logistic Regression Model")

# Create a logistic regression model instance
#------------------------------------------------
model = LogisticRegression(solver='liblinear', random_state=0) 
# Train (fit) the model using the training data
model.fit(X_train_encoded, y_train)

# Make predictions and Model Evaluation
# Make predictions on the test set
y_test_pred = model.predict(X_test_encoded)

# Evaluate the model

conf_matrix_LR = confusion_matrix(y_test, y_test_pred)
print(f"Model Accuracy: {accuracy_score(y_test, y_test_pred)*100:.2f}%")
print("\nConfusion Matrix:\n", conf_matrix_LR)
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))

precision = precision_score(y_test, y_test_pred,pos_label='satisfied')
recall = recall_score(y_test, y_test_pred,pos_label='satisfied')
f1 = f1_score(y_test, y_test_pred,pos_label='satisfied')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# To get predicted probabilities instead of class labels:
probabilities = model.predict_proba(X_test_encoded)
print("Probabilities (first 3 examples):\n", probabilities[:3])

In [ ]:
#  b. Decision Tree Classifier Model 
#-------------------------------------
print("\nDecision Tree Classifier Model")

# Create a Decision Tree classifier object
clf = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=0)
# Train the model using the training sets
clf.fit(X_train_encoded, y_train)

# Make predictions and Model Evaluation
# Make predictions on the test set
y_test_pred = clf.predict(X_test_encoded)

# Evaluate the model
conf_matrix_DT = confusion_matrix(y_test, y_test_pred)
print(f"Model Accuracy: {accuracy_score(y_test, y_test_pred)*100:.2f}%")    
print("\nConfusion Matrix:\n", conf_matrix_DT)
print("\nClassification Report:\n", classification_report(y_test, y_test_pred))

precision = precision_score(y_test, y_test_pred,pos_label='satisfied')
recall = recall_score(y_test, y_test_pred,pos_label='satisfied')
f1 = f1_score(y_test, y_test_pred,pos_label='satisfied')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# To get predicted probabilities instead of class labels:
probabilities = clf.predict_proba(X_test_encoded)
print("\nProbabilities (first 3 examples):\n", probabilities[:3])


In [ ]:
#  c. K Nearest Neighbour (KNN) Classifier Model
#------------------------------------------------
print("\nK Nearest Neighbour (KNN) Classifier Model")

# Find the optimal k value using cross-validation
from sklearn.model_selection import cross_val_score

cv_scores = []
k_values = range(1, 10) # Test k from 1 to 100

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # Perform 5-fold cross-validation
    scores = cross_val_score(knn, X_train_encoded, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Find the optimal k value (the one with the highest mean accuracy)
optimal_k = k_values[np.argmax(cv_scores)]
print(f"The optimal value for k is: {optimal_k}")

# Instantiate the KNN model with the optimal k
knn_optimal = KNeighborsClassifier(n_neighbors=optimal_k)

# Fit the model to the training data
knn_optimal.fit(X_train_encoded, y_train)

# Predict the class for the test set
y_test_pred = knn_optimal.predict(X_test_encoded)

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_test_pred)
print(f'Model Accuracy: {accuracy * 100:.2f}%')

# Print the confusion matrix and classification report
conf_matrix_KNN = confusion_matrix(y_test, y_test_pred)
print("\nConfusion Matrix:")
print(conf_matrix_KNN)

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

precision = precision_score(y_test, y_test_pred,pos_label='satisfied')
recall = recall_score(y_test, y_test_pred,pos_label='satisfied')
f1 = f1_score(y_test, y_test_pred,pos_label='satisfied')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
#  d. Random Forest Classifier Model
#------------------------------------------------
print("\nRandom Forest Classifier Model")

# Initialize the classifier with 100 trees and a random state for reproducibility
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model on the training data
classifier.fit(X_train_encoded, y_train)
y_test_pred = classifier.predict(X_test_encoded)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_test_pred)
print(f'Model Accuracy: {accuracy * 100:.2f}%')

# Generate the confusion matrix
conf_matrix_RF = confusion_matrix(y_test, y_test_pred)
print(f'\nConfusion Matrix:\n{conf_matrix_RF}')

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

precision = precision_score(y_test, y_test_pred,pos_label='satisfied')
recall = recall_score(y_test, y_test_pred,pos_label='satisfied')
f1 = f1_score(y_test, y_test_pred,pos_label='satisfied')

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

In [ ]:
print("""6. Model Evaluation          
    1. Use confusion matrices to analyse model performance 
    2. Select the best model based on evaluation metrics 
""")

# Plotting confusion matrix for Linear Regression Model 
labels = ['satisfied', 'dissatisfied']
cm_df = pd.DataFrame(conf_matrix_LR, index=labels, columns=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='g', cmap='Blues') 
plt.title('Confusion Matrix for Linear Regression Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# Plotting confusion matrix for Decision Tree Classifier Model 
labels = ['satisfied', 'dissatisfied']
cm_df = pd.DataFrame(conf_matrix_DT, index=labels, columns=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='g', cmap='Blues') 
plt.title('Confusion Matrix for Decision Tree Classifier Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# Plotting confusion matrix for K Nearest Neighbours Model 
labels = ['satisfied', 'dissatisfied']
cm_df = pd.DataFrame(conf_matrix_KNN, index=labels, columns=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='g', cmap='Blues') 
plt.title('Confusion Matrix for K Nearest Neighbours Model')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# Plotting confusion matrix for Random Forest Classifier Model 
labels = ['satisfied', 'dissatisfied']
cm_df = pd.DataFrame(conf_matrix_RF, index=labels, columns=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='g', cmap='Blues') 
plt.title('Confusion Matrix for Random Forest Classifier')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
print("""1. Model Performance analysis Using Confusion Matrix - 
-----------------------------------------------------

We read the confusion matrix as follows - 
----------------------------------------
-- True Positive (TP): The model correctly predicted the positive class
-- True Negative (TN): The model correctly predicted the negative class
-- False Positive (FP): The model incorrectly predicted the positive class
-- False Negative (FN): The model incorrectly predicted the negative class 
The goal of a good model is to have high TP and TN values (along the diagonal) and low FP and FN values (off the diagonal). 

1. Confusion matrix for Linear Regression Model. 
TP = 9885               FN = 1724
FP = 1893               TN = 6549

2. Plotting confusion matrix for Decision Tree Classifier Model 
TP = 9885               FN = 1754
FP = 1817               TN = 6625

3. Plotting confusion matrix for K Nearest Neighbours Model 
TP = 10564              FN = 1045
FP = 1594               TN = 6848

4. Plotting confusion matrix for Random Forest Classifier Model
TP = 10741              FN = 868
FP = 1173               TN = 7269 

## Summary - 
The TP and TN are Highest for Random Forest Classifier. 
Similarly, The FP and FN are Lowest for Random Forest Classifier. 
Hence, Random Forest Classifier does the best job in classifying the satisfaction os passengers. 
""")

In [ ]:
print("""2. Select the best model based on evaluation metrics  - 
-------------------------------------------------------
Classification report is shown for all models. It depicts the various evaluation metrics like Precision , Recall, F1 Score. 
On analyzing these metrices, its found that Random Forest Classifier is the best model.

Classification Report for Random Forest Classifier :- 
               precision    recall  f1-score   support

dissatisfied       0.84      0.85      0.85     11609
   satisfied       0.79      0.78      0.78      8442

    accuracy                           0.82     20051
   macro avg       0.82      0.81      0.81     20051
weighted avg       0.82      0.82      0.82     20051
""")


In [ ]:
print("""6. Conclusion        
    1. Summarise findings from the analysis 
    2. Discuss the effectiveness of the chosen model and potential areas for improvement
""")
print("""
#Summary --> 
---------------
1. As part of this exercise, we undertook all activities we need to perform on any new data set. Starting from Data Cleaning, Data Preparation, 
Feature Engineering, EDA, Model training and evaluation
2. We performed the Model training and evaluation for different machine learning models to predict whether a customer is satisfied or dissatisfied. 
3. All models were trained using the same data cleaning, preparation, splitting, encoding and evaluation process to make the comparison process uniform. 
4. We evaluated Logistic Regression, KNN, Decision Tree, and Random Forest models with the abiove process. 
5. Random Forest model gave the best performance with the highest Model accuracy of 89.82%. It also had good balance between Precision and Recall. 

Areas of improvement --> 
--------------------------
I would like to improve on the following aspects - 
  1. Explore better plotting libraries of matplotlib and seaborne for more simpler and appealing plots 
  2. Explore creation of more moduler and reusable functions that can be invoked instead of repeating the code for some of the tasks
  3. Explore pipeline creations for testing and training various models end to end
  4. These steps will increase the code maintainability and readability further
""")